# Intergrowth type: обычные vs тонкие срастания — two methods compared

Runs on a slide-grouped train/test split (no leakage):
- **Method 1** — minerallurgical indices + LDA (Pérez-Barnuevo et al. 2013, reimplemented).
- **Method 2** — ImageNet-pretrained CNN texture classifier (Pérez-Barnuevo 2018 / MISIS 2025 paradigm).

Both trained on your folder labels (рядовая=NORMAL, труднообогатимая=FINE). Talc folders excluded.

In [ ]:
import os, sys, pathlib, torch
# run from the intergrowth/ root so relative data paths resolve
p = pathlib.Path.cwd()
if p.name == 'notebooks':
    os.chdir(p.parent)
sys.path.insert(0, 'src')
print('cwd:', pathlib.Path.cwd())

from intergrowth.data import build_samples, group_split
from intergrowth.constants import CLASS_NAMES
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

samples = build_samples()
train, test = group_split(samples, test_size=0.2, seed=42)
from collections import Counter
print('total:', len(samples), '| classes:', {CLASS_NAMES[k]: v for k, v in Counter(s.label for s in samples).items()})
print('train:', len(train), '| test:', len(test))
assert samples, 'No images found. Symlink data or edit intergrowth/constants.DATA_ROOTS.'

## Method 1 — minerallurgical indices + LDA (Pérez-Barnuevo 2013)

In [ ]:
from intergrowth.lda_model import build_feature_matrix, LdaClassifier
from intergrowth.evaluate import eval_method1, format_result

Xtr, ytr = build_feature_matrix(train)   # GMM phases -> grains -> indices
clf = LdaClassifier.fit(Xtr, ytr)
print('top discriminant features:')
for name, w in clf.coef_table():
    print(f'  {name:>22}: {w:+.3f}')

res1 = eval_method1(clf, test)
print('\n' + format_result(res1))

## Method 2 — ImageNet-pretrained CNN texture classifier

In [ ]:
from torch.utils.data import DataLoader
from intergrowth.tiles import TileDataset
from intergrowth.cnn_model import build_cnn, CnnConfig
from intergrowth.train_cnn import train_cnn, class_weights, CnnTrainConfig
from intergrowth.evaluate import eval_method2, format_result

BATCH = 16 if device.type == 'cuda' else 4
EPOCHS = 8 if device.type == 'cuda' else 1

ds = TileDataset(train, augment=True)
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=4 if device.type=='cuda' else 0, drop_last=True)
model = build_cnn(CnnConfig(backbone='convnext_tiny', pretrained=True))
w = class_weights(train, 2, device)
train_cnn(model, loader, device, w, CnnTrainConfig(epochs=EPOCHS, amp=device.type=='cuda'))

res2 = eval_method2(model, test, device)
print('\n' + format_result(res2))

## Side-by-side

In [ ]:
for r in (res1, res2):
    print(f'{r.method:<34} acc={r.accuracy*100:5.1f}%  balanced={r.balanced_accuracy*100:5.1f}%')
print('\nNOTE: Method 1 is interpretable (grain indices); Method 2 is data-driven texture.')
print('Labels are image-level (folder sort) — this is the honest ceiling of both.')